#Join Strategies

In [0]:
# Databricks notebook source
from pyspark.sql import SparkSession
from pyspark.sql import Row

# Sample Employee info
data_employee = [
    Row(employee_id=1, name='Alice', city='New York'),
    Row(employee_id=2, name='Bob', city='Los Angeles'),
    Row(employee_id=3, name='Charlie', city='Chicago'),
    Row(employee_id=4, name='David', city='Houston'),
    Row(employee_id=5, name='Eva', city='Phoenix')
]

# Sample Salary info
data_salary = [
    Row(employee_id=1, salary=70000, deprt='HR'),
    Row(employee_id=2, salary=80000, deprt='Engineering'),
    Row(employee_id=3, salary=65000, deprt='Finance'),
    Row(employee_id=6, salary=90000, deprt='Marketing'),
    Row(employee_id=7, salary=75000, deprt='Sales')
]

spark = SparkSession.getActiveSession()
df_employee = spark.createDataFrame(data_employee)
df_salary = spark.createDataFrame(data_salary)

# Register temp views for SQL

# Employee info
df_employee.createOrReplaceTempView('employee')
# Salary info
df_salary.createOrReplaceTempView('salary')

display(df_employee)
display(df_salary)

####Broadcast Join Threshold

In [0]:
threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold") #default = 10MB
print(threshold)

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "1gb")
threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
print(threshold)

####Getting Physical Plan

In [0]:
#finds the physical plan
df_employee.join(df_salary, on='employee_id', how='inner').explain()

In [0]:
%sql
EXPLAIN
SELECT *
FROM employee
INNER JOIN salary
ON employee.employee_id = salary.employee_id

####Hint for broadcast join

In [0]:
#A hint to determine if spark is doing the broadcast or not
df=df_employee.join(df_salary.hint("broadcast"), on='employee_id', how='inner')
display(df)

In [0]:
%sql
SELECT /*+BROADCAST(salary) */
  employee.*
, salary.salary
, salary.deprt
FROM employee
INNER JOIN salary
ON employee.employee_id = salary.employee_id